In [1]:
from w2t_bkin.ingest import pose
from pathlib import Path

In [2]:
# We use the script to generate the mock ttl signal form pose data
! python -m pose2ttl --fps=150 --input_file C:\data\W2T_202509_BA\data\interim\MLA-026805\20250905\dlc-pose\example_fileDLC.h5 --output_file ttl_signals_aux.txt

/home/borja/w2t-bkin/.venv/bin/python: No module named pose2ttl


In [3]:
from w2t_bkin.ingest import events

In [4]:
ttl_path = Path("/mnt/c/data/w2t-bkin/data/W2T_202509_BA_old/data/raw/MLA-026805/20250905/TTLs/cue_signals.txt")
ttl_data = events.load_ttl_file(ttl_path)
ttl_data

[73.333333,
 82.08,
 91.34,
 101.006667,
 111.166667,
 123.486667,
 131.193333,
 141.08,
 149.1,
 159.866667,
 170.546667,
 236.133333,
 343.966667,
 354.426667,
 363.733333,
 371.673333,
 379.006667,
 388.486667,
 398.673333,
 407.993333,
 417.286667,
 427.433333,
 437.853333,
 446.953333,
 457.186667,
 466.626667,
 475.553333,
 484.746667,
 492.813333,
 501.926667,
 512.866667,
 520.693333,
 530.28,
 538.28,
 546.726667,
 556.186667,
 564.426667,
 574.313333,
 582.16,
 591.633333,
 601.16,
 609.766667,
 620.406667,
 638.58,
 648.806667,
 658.186667,
 667.533333,
 677.386667,
 686.453333,
 693.406667,
 701.773333,
 711.093333,
 720.713333,
 729.206667,
 737.393333,
 747.406667,
 757.54,
 766.673333,
 774.426667,
 784.886667,
 794.74,
 804.553333,
 815.28,
 823.946667,
 835.52,
 844.693333,
 853.333333,
 863.626667,
 872.946667,
 882.766667,
 890.42,
 899.593333,
 909.806667,
 919.913333,
 928.886667,
 939.073333,
 948.213333,
 957.48,
 966.633333,
 975.0,
 983.42,
 991.726667,
 1001.5

# How to import pose data into nwb

In [5]:
import h5py
import numpy as np

import logging
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple, Union

import h5py
from ndx_pose import PoseEstimation, PoseEstimationSeries, Skeleton, Skeletons
import numpy as np
import pandas as pd
from pynwb import TimeSeries

from w2t_bkin.exceptions import PoseError
from w2t_bkin.utils import derive_bodyparts_from_data, log_missing_keypoints, normalize_keypoints_to_dict



In [16]:
path = Path("/mnt/c/data/w2t-bkin/data/W2T_202509_BA_old/data/interim/MLA-026805/20250905/dlc-pose/example_fileDLC.h5")
pose_data_1 = pose.import_dlc_pose(h5_path=path)
pose_data_1

([{'frame_index': 0,
   'keypoints': {'C1_end': {'name': 'C1_end',
     'x': 845.5932006835938,
     'y': 501.23284912109375,
     'confidence': 0.9999889135360718},
    'C1_end_reverse': {'name': 'C1_end_reverse',
     'x': 113.41797637939453,
     'y': 465.38531494140625,
     'confidence': 0.9999966621398926},
    'C1_mid': {'name': 'C1_mid',
     'x': 785.9649047851562,
     'y': 489.6040344238281,
     'confidence': 0.9999483823776245},
    'C1_mid_reverse': {'name': 'C1_mid_reverse',
     'x': 168.0681915283203,
     'y': 463.1308288574219,
     'confidence': 0.9999599456787109},
    'C1_start': {'name': 'C1_start',
     'x': 724.0962524414062,
     'y': 472.52020263671875,
     'confidence': 0.9999492168426514},
    'C1_start_reverse': {'name': 'C1_start_reverse',
     'x': 224.27008056640625,
     'y': 456.8577880859375,
     'confidence': 0.9999711513519287},
    'C2_end': {'name': 'C2_end',
     'x': 780.0291748046875,
     'y': 414.658447265625,
     'confidence': 0.99999010

In [7]:
PoseEstimationSeries

ndx_pose.pose.PoseEstimationSeries

In [8]:
from w2t_bkin.ingest.pose import KeypointsDict, PoseMetadata
h5_path = Path("/mnt/c/data/w2t-bkin/data/W2T_202509_BA_old/data/interim/MLA-026805/20250905/sleap-pose/example_fileSLEAP.h5")

fps = 150  # CHECK THIS ARE THE SAME AS FOR THE FPS SCRIPT INPUT

with h5py.File(h5_path, "r") as f:
    # Read datasets
    node_names_raw = f["node_names"][:]
    # Decode bytes to strings if necessary
    node_names = [name.decode("utf-8") if isinstance(name, bytes) else str(name) for name in node_names_raw]

    points = f["tracks"][:]  # (frames, instances, nodes, 2)
    scores = f["point_scores"][:]  # (frames, instances, nodes)
    times = np.arange(points.shape[-1]) / fps

    pose_estimation_series = []
    for i, bodypart in enumerate(node_names):
        nwb_pose = PoseEstimationSeries(
            name=bodypart,
            description="something",
            data=points[0,:, i].T,
            unit="pizels",
            reference_frame="(0,0) corresponds to the top-left corner of the video.",
            timestamps=times,
            confidence=scores[0,i],
            confidence_definition=None,
        )
        pose_estimation_series.append(nwb_pose)

In [9]:
pestimation = PoseEstimation(
        name=f"PoseEstimation_Something",
        pose_estimation_series=pose_estimation_series,
    )


In [10]:
from w2t_bkin.ingest import bpod

files = [
    Path("/mnt/c/data/w2t-bkin/data/W2T_202509_BA_old/data/raw/MLA-026805/20250905/Bpod/MLA-026805_(2)_W2T_right_20250905_191956.mat"),
    Path("/mnt/c/data/w2t-bkin/data/W2T_202509_BA_old/data/raw/MLA-026805/20250905/Bpod/MLA-026805_(2)_W2T_right_20250905_192210.mat")
]

bpod_data = bpod.parse_bpod_from_files(files, continuous_time=True)
bpod_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Fri Sep  5 19:21:58 2025',
 '__version__': '1.0',
 '__globals__': [],
 'SessionData': {'Analog': <scipy.io.matlab._mio5_params.mat_struct at 0x7a147f48d420>,
  'Info': <scipy.io.matlab._mio5_params.mat_struct at 0x7a147f48d210>,
  'SettingsFile': <scipy.io.matlab._mio5_params.mat_struct at 0x7a147f48d300>,
  'nTrials': 131,
  'RawEvents': {'Trial': [<scipy.io.matlab._mio5_params.mat_struct at 0x7a147f48d6c0>,
    <scipy.io.matlab._mio5_params.mat_struct at 0x7a147ef96890>]},
  'RawData': <scipy.io.matlab._mio5_params.mat_struct at 0x7a147f48d1b0>,
  'TrialStartTimestamp': [0.1822,
   8.2676,
   15.8586,
   65.7463,
   73.8287,
   82.07,
   89.6723,
   98.0189,
   106.1988,
   114.3413,
   122.4939,
   129.7481,
   138.2821,
   146.2618,
   155.5739,
   163.8435,
   171.6664,
   180.6962,
   188.97140000000002,
   196.42450000000002,
   204.0017,
   211.9613,
   220.6119,
   228.8961,
   236.5427,
   244.4635,
   253.8

# How to get the correct offsets before generating the trials table???

In [11]:
from w2t_bkin.sync import ttl as sync
from types import SimpleNamespace


trial_type_configs = [
    dict(
        trial_type=1,
        sync_signal="the_cue_for_trial_1",
        sync_ttl="cue_ttl",  # NOTE IS THE SAME AS in align_bpod_trials_to_ttl
    ),
    dict(
        trial_type=2,
        sync_signal="the_cue_for_trial_2",
        sync_ttl="cue_ttl",  # NOTE IS THE SAME AS in align_bpod_trials_to_ttl
    ),
]


offsets, _warnings = sync.align_bpod_trials_to_ttl(
    trial_type_configs=trial_type_configs,
    bpod_data=bpod_data,
    ttl_pulses={"cue_ttl": ttl_data}
)

Trial 1: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 2: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 3: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 4: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 5: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 6: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 7: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 8: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 9: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 10: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 11: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 12: sync_signal 'the_cue_for_trial_1' not found or not visited, skipping
Trial 13: sync_signal 'the_cue_for_trial_1' not found or not 

In [12]:
offsets

{}

# Now we can get the trials table

In [13]:
from w2t_bkin.ingest import behavior

task, task_recording, trials_table = behavior.extract_behavioral_data(
    bpod_data,
    trial_offsets=offsets,
    )

/home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/hdmf/container.py:542: UserWarning: The linked table for DynamicTableRegion 'event_type' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()
/home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/hdmf/container.py:542: UserWarning: The linked table for DynamicTableRegion 'state_type' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()
/home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/hdmf/container.py:542: UserWarning: The linked table for DynamicTableRegion 'action_type' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()


In [14]:
trials_table

,start_time,stop_time,states,events,actions
id,,,,,
0,0.1822,8.2376,"[0, 1, 2, 3, 4, 5]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]",[]
1,8.2676,15.8296,"[6, 7, 8, 9, 10, 11]","[24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81]",[]
2,15.8586,23.8634,"[12, 13, 14]","[82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]",[]
3,65.7463,73.7976,"[15, 16, 17]","[100, 101, 102, 103]",[]
